# EuroSAT Land-Use Classification — Kaggle Notebook

**Dataset:** EuroSAT RGB (10-class satellite imagery)
**Model:** EfficientNet-B0 (timm, pretrained on ImageNet)

This notebook is intentionally **thin** — all logic lives in the Python modules
under `src/`.  The notebook only:
1. Installs dependencies
2. Clones / sets up the project
3. Configures the dataset path
4. Calls `train.py` and `evaluate.py`

---
**Before running:** Enable GPU in `Settings → Accelerator → GPU T4 x2` (or P100)

## 1. Environment check

In [ ]:
import subprocess, sys, os

# Verify GPU is available
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU detected. Training will be very slow.")
    print("Go to Settings → Accelerator → GPU T4 x2 and restart the kernel.")

## 2. Install dependencies

Kaggle already has PyTorch + torchvision pre-installed.  
We only need to install `timm` and a few extras.

In [ ]:
# Install missing packages (timm, tqdm already often available on Kaggle)
!pip install -q timm>=0.9.12 tqdm scikit-learn PyYAML

## 3. Set up the project

**Option A — Upload the repo as a Kaggle Dataset (recommended)**
1. Zip the entire `Kaggle_EuroSAT_Classification/` folder
2. Upload to Kaggle → Datasets → New Dataset
3. Attach it to this notebook as an input
4. It will appear at `/kaggle/input/<your-dataset-name>/`

**Option B — Paste files directly** (see KAGGLE_GUIDE.md for details)

The cell below handles both options.  Set `REPO_PATH` to wherever your files are.

In [ ]:
import os, sys
from pathlib import Path

# ── Find the project root ─────────────────────────────────────────────────────
# Adjust this path to match where you uploaded the repo
POSSIBLE_ROOTS = [
    "/kaggle/input/eurosat-ml-project/Kaggle_EuroSAT_Classification",  # uploaded as dataset
    "/kaggle/working/Kaggle_EuroSAT_Classification",                   # cloned / extracted
    "/kaggle/working",                                      # files copied directly
]

REPO_ROOT = None
for p in POSSIBLE_ROOTS:
    if Path(p).exists() and (Path(p) / "src").exists():
        REPO_ROOT = p
        break

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not find project root. "
        "Please check POSSIBLE_ROOTS above and set REPO_ROOT manually."
    )

print(f"Project root: {REPO_ROOT}")
os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)

# Quick sanity check
from src.utils.config import load_config
print("✓ src package importable")

## 4. Verify dataset path

The EuroSAT dataset must be attached as a Kaggle input.  
It should have this structure:
```
/kaggle/input/eurosat-dataset/
└── EuroSAT/
    ├── AnnualCrop/      (2000+ .jpg files)
    ├── Forest/
    ├── HerbaceousVegetation/
    ├── Highway/
    ├── Industrial/
    ├── Pasture/
    ├── PermanentCrop/
    ├── Residential/
    ├── River/
    └── SeaLake/
```

In [ ]:
from pathlib import Path

# ── Locate dataset ────────────────────────────────────────────────────────────
DATASET_CANDIDATES = [
    "/kaggle/input/eurosat-dataset/EuroSAT",
    "/kaggle/input/eurosat/EuroSAT",
    "/kaggle/input/land-use-classification/EuroSAT",
    "/kaggle/input/eurosat-rgb/EuroSAT",
]

DATASET_PATH = None
for candidate in DATASET_CANDIDATES:
    if Path(candidate).exists():
        DATASET_PATH = candidate
        break

if DATASET_PATH is None:
    # List what is available so the user can find the right path
    print("EuroSAT not found at expected paths. Available Kaggle inputs:")
    for d in sorted(Path("/kaggle/input").iterdir()):
        print(f"  {d}")
    raise RuntimeError(
        "Dataset not found. Add EuroSAT as an input dataset and update DATASET_PATH."
    )

print(f"Dataset found at: {DATASET_PATH}")

# Count files per class
total = 0
for cls_dir in sorted(Path(DATASET_PATH).iterdir()):
    if cls_dir.is_dir():
        n = len(list(cls_dir.glob("*")))
        print(f"  {cls_dir.name:<30} {n:>5} images")
        total += n
print(f"  {'TOTAL':<30} {total:>5} images")

## 5. Configure training

We patch the config to use the Kaggle dataset path and optimise for Kaggle's GPU.

In [ ]:
# Build the CLI override list
OVERRIDES = [
    f"data.dataset_path={DATASET_PATH}",
    "training.epochs=30",
    "training.batch_size=64",
    "model.architecture=efficientnet_b0",
    "model.pretrained=true",
    "training.mixed_precision=true",
    "data.num_workers=2",       # Kaggle limits worker processes
    "project.output_dir=/kaggle/working/outputs",
    "training.checkpoint_dir=/kaggle/working/outputs/checkpoints",
    "evaluation.output_dir=/kaggle/working/outputs/evaluation",
]

# Quick sanity-load
from src.utils.config import load_config
cfg = load_config("configs/config.yaml", overrides=OVERRIDES)
print(f"Architecture : {cfg.model.architecture}")
print(f"Epochs       : {cfg.training.epochs}")
print(f"Batch size   : {cfg.training.batch_size}")
print(f"Dataset path : {cfg.data.dataset_path}")
print(f"Output dir   : {cfg.project.output_dir}")

## 6. Start training

In [ ]:
# Build the override string for subprocess call
override_args = " ".join(f'--override "{o}"' for o in OVERRIDES)

cmd = f"python scripts/train.py --config configs/config.yaml {override_args}"
print("Running:", cmd)
print("-" * 70)

import subprocess, sys
result = subprocess.run(
    cmd, shell=True, cwd=REPO_ROOT, check=False
)
if result.returncode != 0:
    print("\nTraining exited with non-zero code. Check the log above for errors.")
else:
    print("\n✓ Training completed successfully!")

## 7. View training results

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

output_dir = Path("/kaggle/working/outputs")

# ── Training history ──────────────────────────────────────────────────────────
history_path = output_dir / "history.json"
if history_path.exists():
    history = json.loads(history_path.read_text())
    epochs = len(history["val_acc"])
    best_acc = max(history["val_acc"])
    best_epoch = history["val_acc"].index(best_acc) + 1
    print(f"Epochs trained      : {epochs}")
    print(f"Best val accuracy   : {best_acc:.4f} (epoch {best_epoch})")
    print(f"Final train accuracy: {history['train_acc'][-1]:.4f}")

# ── Metrics ───────────────────────────────────────────────────────────────────
metrics_path = output_dir / "evaluation" / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print("\n── Test Set Metrics ──")
    print(f"  Accuracy   : {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
    print(f"  F1 Macro   : {metrics['f1_macro']:.4f}")
    print(f"  F1 Weighted: {metrics['f1_weighted']:.4f}")

In [ ]:
# Display training curves
curves_path = output_dir / "evaluation" / "training_curves.png"
if curves_path.exists():
    display(Image(str(curves_path)))

In [ ]:
# Display confusion matrix
cm_path = output_dir / "evaluation" / "confusion_matrix.png"
if cm_path.exists():
    display(Image(str(cm_path)))

In [ ]:
# Display per-class F1
f1_path = output_dir / "evaluation" / "f1_per_class.png"
if f1_path.exists():
    display(Image(str(f1_path)))

## 8. Save outputs for download

Anything saved to `/kaggle/working/` can be downloaded from the notebook output panel.

In [ ]:
import shutil
from pathlib import Path

working = Path("/kaggle/working")

print("Files available for download:")
for f in sorted(working.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {str(f.relative_to(working)):60s}  {size_kb:8.1f} KB")

## 9. (Optional) Try a different model

Change the cell below and re-run from step 5 to experiment with different architectures.

In [ ]:
# Example: switch to a Vision Transformer
# OVERRIDES_VIT = OVERRIDES.copy()
# OVERRIDES_VIT = [o.replace("efficientnet_b0", "vit_small_patch16_224") for o in OVERRIDES_VIT]
# OVERRIDES_VIT.append("training.batch_size=32")
# OVERRIDES_VIT.append("training.learning_rate=3e-4")
# override_args = " ".join(f'--override "{o}"' for o in OVERRIDES_VIT)
# !python scripts/train.py --config configs/config.yaml {override_args}
print("Uncomment the lines above to train a ViT model.")